In [49]:
import pandas as pd

NJ_SNF = pd.read_csv(
    "data/NJ_SNF.csv",
    dtype={"CMS Certification Number (CCN)": str}
)

NY_SNF = pd.read_csv(
    "data/NY_SNF.csv",
    dtype={"CMS Certification Number (CCN)": str}
)

In [50]:
# --- function to reduce a raw SNF file to one row per facility in selected counties ---
def build_metro_roster(df, allowed_counties):
    keep_cols = [
        "CMS Certification Number (CCN)",
        "Provider Name",
        "City/Town",
        "County/Parish",
    ]

    out = df[keep_cols].copy()

    out["CMS Certification Number (CCN)"] = (
        out["CMS Certification Number (CCN)"]
        .astype(str)
        .str.strip()
        .str.zfill(6)
    )

    out["Provider Name"] = out["Provider Name"].astype(str).str.strip()
    out["City/Town"] = out["City/Town"].astype(str).str.strip()
    out["County/Parish"] = (
        out["County/Parish"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    out = out[out["County/Parish"].isin(allowed_counties)].copy()

    out = (
        out.sort_values(
            ["County/Parish", "City/Town", "Provider Name", "CMS Certification Number (CCN)"]
        )
        .drop_duplicates(subset=["CMS Certification Number (CCN)"], keep="first")
        .reset_index(drop=True)
    )

    return out

In [51]:
# I'd like to clarify my interpretation of "NYC Metropolitan Area" in the prompt since there are various definitions of it that float around. The following counties are defined in a report published by the White House in 2023 which delineates/defines all the major US metropolitan areas, as defined under code #35620: "NY-NJ Metropolitan Statistical Area".
# Link: https://www.whitehouse.gov/wp-content/uploads/2023/07/OMB-Bulletin-23-01.pdf

NJ_NYC_METRO_COUNTIES = {
    "Bergen",
    "Hudson",
    "Passaic",
    "Essex",
    "Hunterdon",
    "Morris",
    "Sussex",
    "Union",
    "Middlesex",
    "Monmouth",
    "Ocean",
    "Somerset",
}

NY_NYC_METRO_COUNTIES = {
    "Bronx",
    "Kings",
    "New York",
    "Queens",
    "Richmond",
    "Nassau",
    "Suffolk",
    "Putnam",
    "Rockland",
    "Westchester",
}

In [52]:
NJ_roster = build_metro_roster(NJ_SNF, NJ_NYC_METRO_COUNTIES)
NY_roster = build_metro_roster(NY_SNF, NY_NYC_METRO_COUNTIES)

NJ_roster["State"] = "NJ"
NY_roster["State"] = "NY"

metro_roster = pd.concat([NJ_roster, NY_roster], ignore_index=True)

metro_roster = metro_roster[
    [
        "CMS Certification Number (CCN)",
        "Provider Name",
        "City/Town",
        "County/Parish",
        "State",
    ]
].sort_values(
    ["State", "County/Parish", "City/Town", "Provider Name"]
).reset_index(drop=True)

display(metro_roster.head(5))
print("Total facilities:", len(metro_roster))
print("Unique CCNs:", metro_roster["CMS Certification Number (CCN)"].nunique())

,CMS Certification Number (CCN),Provider Name,City/Town,County/Parish,State
0,315497,ALLENDALE REHABILITATION AND HEALTHCARE CENTER,ALLENDALE,Bergen,NJ
1,315313,CAREONE AT CRESSKILL,CRESSKILL,Bergen,NJ
2,315360,Emerson Health Care Center,EMERSON,Bergen,NJ
3,315377,"ACTORS FUND HOME, THE",ENGLEWOOD,Bergen,NJ
4,315349,"COMPLETE CARE AT INGLEMOOR, LLC",ENGLEWOOD,Bergen,NJ


Total facilities: 551
Unique CCNs: 551


In [53]:
# ----------------------------
# 1. load Provider Information
# ----------------------------
provider_path = "data/NH_ProviderInfo_Mar2026.csv"
provider_df = pd.read_csv(provider_path)

# ----------------------------
# 2. normalize CCN in both dfs
# ----------------------------
metro_roster["CMS Certification Number (CCN)"] = (
    metro_roster["CMS Certification Number (CCN)"]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

provider_df["CMS Certification Number (CCN)"] = (
    provider_df["CMS Certification Number (CCN)"]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

# ----------------------------
# 3. keep only roster-matching CCNs
# ----------------------------
valid_ccns = set(metro_roster["CMS Certification Number (CCN)"])
provider_df = provider_df[
    provider_df["CMS Certification Number (CCN)"].isin(valid_ccns)
].copy()

# ----------------------------
# 4. keep only needed columns
# ----------------------------
cols_needed = [
    "CMS Certification Number (CCN)",
    "Health Inspection Rating",
    "Short-Stay QM Rating",
    "Adjusted Total Nurse Staffing Hours per Resident per Day",
    "Number of Certified Beds",
    "Average Number of Residents per Day",
]

provider_df = provider_df[cols_needed].copy()

# ----------------------------
# 5. convert numeric columns
# ----------------------------
numeric_cols = [
    "Health Inspection Rating",
    "Short-Stay QM Rating",
    "Adjusted Total Nurse Staffing Hours per Resident per Day",
    "Number of Certified Beds",
    "Average Number of Residents per Day",
]

for col in numeric_cols:
    provider_df[col] = pd.to_numeric(provider_df[col], errors="coerce")

# ----------------------------
# 6. derive occupancy proxy
# ----------------------------
provider_df["Estimated Occupancy Rate"] = (
        provider_df["Average Number of Residents per Day"] /
        provider_df["Number of Certified Beds"]
)

# optional: keep within sensible bounds if data glitches exist
provider_df["Estimated Occupancy Rate"] = provider_df["Estimated Occupancy Rate"].clip(lower=0)

# ----------------------------
# 7. rename to cleaner labels
# ----------------------------
provider_df = provider_df.rename(columns={
    "Health Inspection Rating": "health_inspection_rating",
    "Short-Stay QM Rating": "short_stay_qm_rating",
    "Adjusted Total Nurse Staffing Hours per Resident per Day": "adjusted_total_nurse_hours_per_resident_day",
    "Estimated Occupancy Rate": "estimated_occupancy_rate",
})

# keep exactly the 4 added metrics
provider_df = provider_df[
    [
        "CMS Certification Number (CCN)",
        "health_inspection_rating",
        "short_stay_qm_rating",
        "adjusted_total_nurse_hours_per_resident_day",
        "estimated_occupancy_rate",
    ]
].copy()

# ----------------------------
# 8. merge into roster
# ----------------------------
roster_enriched_df = metro_roster.merge(
    provider_df,
    on="CMS Certification Number (CCN)",
    how="left"
)

# ----------------------------
# 9. inspect + save
# ----------------------------
display(roster_enriched_df.head(200))

print("Rows in final table:", len(roster_enriched_df))
print("Unique CCNs in final table:", roster_enriched_df["CMS Certification Number (CCN)"].nunique())

roster_enriched_df.to_csv("roster_with_positive_metrics.csv", index=False)

,CMS Certification Number (CCN),Provider Name,City/Town,County/Parish,State,health_inspection_rating,short_stay_qm_rating,adjusted_total_nurse_hours_per_resident_day,estimated_occupancy_rate
0,315497,ALLENDALE REHABILITATION AND HEALTHCARE CENTER,ALLENDALE,Bergen,NJ,3.0,4.0,3.25771,0.829167
1,315313,CAREONE AT CRESSKILL,CRESSKILL,Bergen,NJ,4.0,5.0,3.15164,0.737168
2,315360,Emerson Health Care Center,EMERSON,Bergen,NJ,5.0,4.0,4.33627,0.876774
3,315377,"ACTORS FUND HOME, THE",ENGLEWOOD,Bergen,NJ,4.0,2.0,5.01175,0.872897
4,315349,"COMPLETE CARE AT INGLEMOOR, LLC",ENGLEWOOD,Bergen,NJ,2.0,2.0,3.35536,0.933871
...,...,...,...,...,...,...,...,...,...
195,315085,COMPLETE CARE AT CHESTNUT HILL LLC,PASSAIC,Passaic,NJ,1.0,3.0,3.12677,0.940541
196,315221,"COMPLETE CARE AT HAMILTON, LLC",PASSAIC,Passaic,NJ,2.0,2.0,3.84048,0.867500
197,315507,"Barnert Subacute Rehabilitation Center, Llc",PATERSON,Passaic,NJ,4.0,1.0,4.12586,0.910294
198,315331,COMPLETE CARE AT FAIR LAWN EDGE,PATERSON,Passaic,NJ,3.0,1.0,3.87380,0.970556


Rows in final table: 551
Unique CCNs in final table: 551


In [54]:
df = roster_enriched_df.copy()

# optional cleanup so grouping is consistent
df["State"] = df["State"].astype(str).str.strip().str.upper()
df["County/Parish"] = df["County/Parish"].astype(str).str.strip()
df["City/Town"] = df["City/Town"].astype(str).str.strip()

for state in ["NJ", "NY"]:
    print(f"\n{'='*60}")
    print(f"STATE: {state}")
    print(f"{'='*60}")

    state_df = df[df["State"] == state].copy()

    # top 3 counties by number of facilities
    county_counts = (
        state_df.groupby("County/Parish")["CMS Certification Number (CCN)"]
        .count()
        .sort_values(ascending=False)
    )

    top_3_counties = county_counts.head(3)

    print("\nTop 3 counties by number of SNFs:")
    for rank, (county, count) in enumerate(top_3_counties.items(), start=1):
        print(f"{rank}. {county}: {count} facilities")

    print("\nTop 3 towns within each of those counties:")
    for county in top_3_counties.index:
        print(f"\n{county} County:")

        town_counts = (
            state_df[state_df["County/Parish"] == county]
            .groupby("City/Town")["CMS Certification Number (CCN)"]
            .count()
            .sort_values(ascending=False)
            .head(3)
        )

        for rank, (town, count) in enumerate(town_counts.items(), start=1):
            print(f"   {rank}. {town}: {count} facilities")


STATE: NJ

Top 3 counties by number of SNFs:
1. Monmouth: 33 facilities
2. Essex: 32 facilities
3. Ocean: 32 facilities

Top 3 towns within each of those counties:

Monmouth County:
   1. FREEHOLD: 5 facilities
   2. WALL: 4 facilities
   3. NEPTUNE: 3 facilities

Essex County:
   1. CEDAR GROVE: 5 facilities
   2. NEWARK: 5 facilities
   3. WEST ORANGE: 5 facilities

Ocean County:
   1. TOMS RIVER: 9 facilities
   2. LAKEWOOD: 5 facilities
   3. BRICK: 4 facilities

STATE: NY

Top 3 counties by number of SNFs:
1. Queens: 57 facilities
2. Bronx: 44 facilities
3. Westchester: 42 facilities

Top 3 towns within each of those counties:

Queens County:
   1. FAR ROCKAWAY: 11 facilities
   2. FLUSHING: 11 facilities
   3. JAMAICA: 7 facilities

Bronx County:
   1. BRONX: 41 facilities
   2. RIVERDALE: 3 facilities

Westchester County:
   1. NEW ROCHELLE: 6 facilities
   2. YONKERS: 5 facilities
   3. WHITE PLAINS: 3 facilities


In [55]:
for state in ["NJ", "NY"]:
    state_df = roster_enriched_df[roster_enriched_df["State"] == state].copy()

    county_counts = (
        state_df.groupby("County/Parish")["CMS Certification Number (CCN)"]
        .count()
        .sort_values(ascending=False)
    )

    top3_total = county_counts.head(3).sum()
    state_total = len(state_df)
    pct = 100 * top3_total / state_total

    print(f"{state}:")
    print(f"  Total SNFs in state = {state_total}")
    print(f"  SNFs in top 3 counties = {top3_total}")
    print(f"  Share held by top 3 counties = {pct:.2f}%\n")

NJ:
  Total SNFs in state = 252
  SNFs in top 3 counties = 97
  Share held by top 3 counties = 38.49%

NY:
  Total SNFs in state = 299
  SNFs in top 3 counties = 143
  Share held by top 3 counties = 47.83%



In [56]:
# Mean and variance of estimated occupancy rate across the full 551-facility dataset

occ = pd.to_numeric(
    roster_enriched_df["estimated_occupancy_rate"],
    errors="coerce"
).dropna()

mean_occ = occ.mean()
sample_var_occ = occ.var()        # pandas default: sample variance, ddof=1
population_var_occ = occ.var(ddof=0)

print(f"Facility count used: {len(occ)}")
print(f"Mean estimated occupancy rate: {mean_occ:.6f}")
print(f"Sample variance of estimated occupancy rate: {sample_var_occ:.6f}")
print(f"Population variance of estimated occupancy rate: {population_var_occ:.6f}")

Facility count used: 547
Mean estimated occupancy rate: 0.865419
Sample variance of estimated occupancy rate: 0.015041
Population variance of estimated occupancy rate: 0.015014


In [57]:
import numpy as np
import pandas as pd

occ = pd.to_numeric(roster_enriched_df["estimated_occupancy_rate"], errors="coerce").dropna()

mean_occ = occ.mean()
sd_occ = occ.std()  # sample standard deviation

lower = mean_occ - sd_occ
upper = mean_occ + sd_occ

pct_within_1sd = ((occ >= lower) & (occ <= upper)).mean() * 100

print(f"Mean: {mean_occ:.6f}")
print(f"Sample SD: {sd_occ:.6f}")
print(f"1-SD interval: [{lower:.6f}, {upper:.6f}]")
print(f"Percent of facilities within 1 SD: {pct_within_1sd:.2f}%")

Mean: 0.865419
Sample SD: 0.122642
1-SD interval: [0.742776, 0.988061]
Percent of facilities within 1 SD: 82.82%


In [58]:
import pandas as pd
import numpy as np

provider_path = "data/NH_ProviderInfo_Mar2026.csv"
provider_bad_df = pd.read_csv(
    provider_path,
    dtype={"CMS Certification Number (CCN)": str}
)

provider_bad_df.columns = (
    provider_bad_df.columns
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

roster_enriched_df["CMS Certification Number (CCN)"] = (
    roster_enriched_df["CMS Certification Number (CCN)"]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

provider_bad_df["CMS Certification Number (CCN)"] = (
    provider_bad_df["CMS Certification Number (CCN)"]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)

valid_ccns = set(roster_enriched_df["CMS Certification Number (CCN)"])

provider_bad_df = provider_bad_df[
    provider_bad_df["CMS Certification Number (CCN)"].isin(valid_ccns)
].copy()

bad_cols = [
    "CMS Certification Number (CCN)",
    "Abuse Icon",
    "Total Weighted Health Survey Score",
    "Rating Cycle 1 Total Health Score",
]

provider_bad_df = provider_bad_df[bad_cols].copy()

provider_bad_df["abuse_icon_flag"] = (
    provider_bad_df["Abuse Icon"]
    .astype(str)
    .str.strip()
    .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    .str.upper()
    .map({"Y": 1, "N": 0})
)

provider_bad_df["total_weighted_health_survey_score"] = pd.to_numeric(
    provider_bad_df["Total Weighted Health Survey Score"],
    errors="coerce"
)

provider_bad_df["rating_cycle_1_total_health_score"] = pd.to_numeric(
    provider_bad_df["Rating Cycle 1 Total Health Score"],
    errors="coerce"
)

provider_bad_df = (
    provider_bad_df[
        [
            "CMS Certification Number (CCN)",
            "abuse_icon_flag",
            "total_weighted_health_survey_score",
            "rating_cycle_1_total_health_score",
        ]
    ]
    .drop_duplicates(subset=["CMS Certification Number (CCN)"], keep="first")
    .reset_index(drop=True)
)

roster_full_df = roster_enriched_df.merge(
    provider_bad_df,
    on="CMS Certification Number (CCN)",
    how="left"
)

display(roster_full_df.head(50))

print("Rows in final table:", len(roster_full_df))
print("Unique CCNs:", roster_full_df["CMS Certification Number (CCN)"].nunique())
print("Non-null Rating Cycle 1 Total Health Score:", roster_full_df["rating_cycle_1_total_health_score"].notna().sum())

roster_full_df.to_csv("roster_with_positive_and_negative_metrics.csv", index=False)

,CMS Certification Number (CCN),Provider Name,City/Town,County/Parish,State,health_inspection_rating,short_stay_qm_rating,adjusted_total_nurse_hours_per_resident_day,estimated_occupancy_rate,abuse_icon_flag,total_weighted_health_survey_score,rating_cycle_1_total_health_score
0,315497,ALLENDALE REHABILITATION AND HEALTHCARE CENTER,ALLENDALE,Bergen,NJ,3.0,4.0,3.25771,0.829167,0,56.00,64.0
1,315313,CAREONE AT CRESSKILL,CRESSKILL,Bergen,NJ,4.0,5.0,3.15164,0.737168,0,31.00,36.0
2,315360,Emerson Health Care Center,EMERSON,Bergen,NJ,5.0,4.0,4.33627,0.876774,0,2.00,0.0
3,315377,"ACTORS FUND HOME, THE",ENGLEWOOD,Bergen,NJ,4.0,2.0,5.01175,0.872897,0,35.00,44.0
4,315349,"COMPLETE CARE AT INGLEMOOR, LLC",ENGLEWOOD,Bergen,NJ,2.0,2.0,3.35536,0.933871,0,94.00,100.0
5,315328,MAPLE GLEN CENTER,FAIRLAWN,Bergen,NJ,3.0,4.0,3.32961,0.757233,0,46.00,60.0
6,315152,CAREONE AT WELLINGTON,HACKENSACK,Bergen,NJ,2.0,5.0,3.37655,0.828906,0,86.00,100.0
7,315460,COMPLETE CARE AT PROSPECT HEIGHTS LLC,HACKENSACK,Bergen,NJ,1.0,5.0,2.81068,0.612755,0,112.00,120.0
8,315295,COMPLETE CARE AT REGENT LLC,HACKENSACK,Bergen,NJ,2.0,3.0,3.36611,0.943333,0,89.00,108.0
9,315386,ATLAS REHABILITATION AND HEALTHCARE AT MAYWOOD,MAYWOOD,Bergen,NJ,3.0,5.0,3.13147,0.925000,0,58.00,76.0


Rows in final table: 551
Unique CCNs: 551
Non-null Rating Cycle 1 Total Health Score: 550


In [59]:

df = roster_full_df.copy()

# Or load from file:
# df = pd.read_csv(
#     "roster_with_positive_and_negative_metrics.csv",
#     dtype={"CMS Certification Number (CCN)": str}
# )

# --------------------------------------------------
# 1. MANUAL WEIGHTS: edit only these numbers
# --------------------------------------------------
W_HEALTH_INSPECTION = 1.0
W_SHORT_STAY_QM = 2.0
W_NURSE_HOURS = 1.0
W_OCCUPANCY = 3.0
W_ABUSE = 5.0
W_CYCLE1_HEALTH_SCORE = 1.0
W_WEIGHTED_HEALTH_SURVEY = 1.0

# --------------------------------------------------
# 2. metric directions
# --------------------------------------------------
metric_directions = {
    "health_inspection_rating": "higher_better",
    "short_stay_qm_rating": "higher_better",
    "adjusted_total_nurse_hours_per_resident_day": "higher_better",
    "estimated_occupancy_rate": "lower_better",
    "abuse_icon_flag": "lower_better",  # 0 better than 1
    "rating_cycle_1_total_health_score": "lower_better",
    "total_weighted_health_survey_score": "lower_better",
}

# --------------------------------------------------
# 3. metrics used in scoring / disqualification
# --------------------------------------------------
scoring_metrics = list(metric_directions.keys())

# --------------------------------------------------
# 4. disqualify facilities with 2+ missing metrics
# --------------------------------------------------
df["missing_metric_count"] = df[scoring_metrics].isna().sum(axis=1)
df["disqualified_top10_flag"] = (df["missing_metric_count"] >= 2)


# --------------------------------------------------
# 5. normalization helper
# --------------------------------------------------
def minmax_normalize(series):
    s = pd.to_numeric(series, errors="coerce")
    s_min = s.min(skipna=True)
    s_max = s.max(skipna=True)

    if pd.isna(s_min) or pd.isna(s_max) or s_min == s_max:
        return pd.Series(np.nan, index=s.index)

    return (s - s_min) / (s_max - s_min)


# --------------------------------------------------
# 6. create normalized component scores
#    conservative rule: missing metric gets component score 0
# --------------------------------------------------
for metric, direction in metric_directions.items():
    norm = minmax_normalize(df[metric])

    if direction == "higher_better":
        component = norm
    else:
        component = 1 - norm

    df[f"{metric}_component"] = component.fillna(0)

# --------------------------------------------------
# 7. weighted composite score
# --------------------------------------------------
weighted_sum = (
        W_HEALTH_INSPECTION * df["health_inspection_rating_component"] +
        W_SHORT_STAY_QM * df["short_stay_qm_rating_component"] +
        W_NURSE_HOURS * df["adjusted_total_nurse_hours_per_resident_day_component"] +
        W_OCCUPANCY * df["estimated_occupancy_rate_component"] +
        W_ABUSE * df["abuse_icon_flag_component"] +
        W_CYCLE1_HEALTH_SCORE * df["rating_cycle_1_total_health_score_component"] +
        W_WEIGHTED_HEALTH_SURVEY * df["total_weighted_health_survey_score_component"]
)

total_weight = (
        W_HEALTH_INSPECTION +
        W_SHORT_STAY_QM +
        W_NURSE_HOURS +
        W_OCCUPANCY +
        W_ABUSE +
        W_CYCLE1_HEALTH_SCORE +
        W_WEIGHTED_HEALTH_SURVEY
)

df["composite_score"] = weighted_sum / total_weight

# --------------------------------------------------
# 8. ranking
#    disqualified facilities stay in the table,
#    but are forced below all eligible facilities
# --------------------------------------------------
df["eligible_for_top10"] = ~df["disqualified_top10_flag"]

df = df.sort_values(
    by=[
        "eligible_for_top10",
        "composite_score",
        "health_inspection_rating",
        "short_stay_qm_rating",
    ],
    ascending=[False, False, False, False]
).reset_index(drop=True)

df["overall_rank"] = np.arange(1, len(df) + 1)

# eligible-only rank
df["eligible_rank"] = np.nan
eligible_mask = df["eligible_for_top10"]
df.loc[eligible_mask, "eligible_rank"] = np.arange(1, eligible_mask.sum() + 1)

# --------------------------------------------------
# 9. top 10 eligible SNFs only
# --------------------------------------------------
top_10_snfs = df.loc[
    df["eligible_for_top10"],
    [
        "eligible_rank",
        "CMS Certification Number (CCN)",
        "Provider Name",
        "City/Town",
        "County/Parish",
        "State",
        "composite_score",
        "missing_metric_count",
        "health_inspection_rating",
        "short_stay_qm_rating",
        "adjusted_total_nurse_hours_per_resident_day",
        "estimated_occupancy_rate",
        "abuse_icon_flag",
        "rating_cycle_1_total_health_score",
        "total_weighted_health_survey_score",
    ]
].head(10).copy()

top_10_snfs = top_10_snfs.rename(columns={"eligible_rank": "rank"})

# --------------------------------------------------
# 10. inspect + save
# --------------------------------------------------
display(top_10_snfs)

print("Total facilities:", len(df))
print("Eligible for top 10:", df["eligible_for_top10"].sum())
print("Disqualified from top 10:", df["disqualified_top10_flag"].sum())

top_10_snfs.to_csv("top_10_snfs_ranked.csv", index=False)
df.to_csv("all_snfs_ranked.csv", index=False)

,rank,CMS Certification Number (CCN),Provider Name,City/Town,County/Parish,State,composite_score,missing_metric_count,health_inspection_rating,short_stay_qm_rating,adjusted_total_nurse_hours_per_resident_day,estimated_occupancy_rate,abuse_icon_flag,rating_cycle_1_total_health_score,total_weighted_health_survey_score
0,1.0,335269,The Wartburg Home,MOUNT VERNON,Westchester,NY,0.918007,0,3.0,5.0,4.70319,0.207619,0,40.0,37.0
1,2.0,335020,HEBREW HOME FOR THE AGED AT RIVERDALE,RIVERDALE,Bronx,NY,0.891388,0,5.0,5.0,3.27139,0.432028,0,0.0,10.0
2,3.0,315029,DAUGHTERS OF ISRAEL PLEASANT VALLEY HOME,WEST ORANGE,Essex,NJ,0.880125,0,3.0,5.0,4.18611,0.316502,0,64.0,60.0
3,4.0,315292,"APPLEWOOD VILLAGE, INC",FREEHOLD,Monmouth,NJ,0.862173,0,4.0,5.0,6.14343,0.596667,0,20.0,15.0
4,5.0,315479,CAREONE AT LIVINGSTON,LIVINGSTON,Essex,NJ,0.857233,0,4.0,5.0,3.92088,0.505000,0,28.0,29.0
5,6.0,335631,CHAPIN HOME FOR THE AGING,JAMAICA,Queens,NY,0.852124,0,5.0,5.0,3.91511,0.619545,0,8.0,7.0
6,7.0,335621,UNITED HEBREW GERIATRIC CENTER,NEW ROCHELLE,Westchester,NY,0.842346,0,4.0,5.0,3.75851,0.571769,0,16.0,14.0
7,8.0,315153,"MANOR, THE",FREEHOLD,Monmouth,NJ,0.837108,0,4.0,4.0,4.31780,0.458537,0,32.0,25.0
8,9.0,335833,JEFFERSON'S FERRY,SOUTH SETAUKET,Suffolk,NY,0.828587,0,5.0,5.0,5.03830,0.773333,0,0.0,4.0
9,10.0,315512,HOBOKEN UNIVERSITY MEDICAL CENTER TCU,HOBOKEN,Hudson,NJ,0.827865,0,4.0,4.0,7.40962,0.646667,0,20.0,20.0


Total facilities: 551
Eligible for top 10: 543
Disqualified from top 10: 8
